# GUN 6: LangChain & Otonom AI Ajanlari

**Egitim:** Ileri Yapay Zeka Gelistirici Programi | **Seviye:** Orta | **Sure:** ~3 saat

---

**Bu Notebook'ta Neler Yapacagiz?**

1. Agent framework'unu sifirdan yazma (ReAct)
2. Custom tool olusturma
3. LangChain chain'leri ve LCEL
4. LangChain agent ile tool kullanimi
5. Memory yonetimi (konusma gecmisi)
6. RAG Agent (dunku RAG + agent birlesmesi)
7. Plan-and-Execute agent
8. Multi-tool akilli asistan
9. Agent debugging ve guvenlik
10. Egzersizler

---
## BOLUM 1: Kurulum

In [ ]:
!pip install langchain==0.2.6 langchain-community==0.2.6 langchain-huggingface==0.0.3 -q
!pip install chromadb==0.5.0 sentence-transformers==3.0.1 -q
!pip install transformers==4.42.3 accelerate numexpr -q
!pip install wikipedia-api -q

import warnings
warnings.filterwarnings("ignore")
print("Kurulum tamamlandi!")

In [ ]:
# LLM yukleme
from transformers import pipeline as hf_pipeline

print("LLM yukleniyor (flan-t5-base)...")
llm_pipe = hf_pipeline("text2text-generation", model="google/flan-t5-base", max_new_tokens=256)

def ask_llm(prompt, max_tokens=256):
    """LLM'e soru sor"""
    result = llm_pipe(prompt, max_new_tokens=max_tokens)[0]["generated_text"]
    return result.strip()

# Test
print(f"LLM test: {ask_llm('What is 2+2?')}")
print("Hazir!")

---
## BOLUM 2: ReAct Agent Sifirdan

Hicbir framework kullanmadan, ReAct agent'ini sifirdan yazacagiz.
Boylece agent'in icini gercekten anlayacaksin.

In [ ]:
import re
import math
import json
from datetime import datetime

# ============================================
# ADIM 1: Tool'lari tanimla
# ============================================

class Tool:
    def __init__(self, name, description, func):
        self.name = name
        self.description = description
        self.func = func
    
    def run(self, input_text):
        try:
            return str(self.func(input_text))
        except Exception as e:
            return f"Hata: {str(e)}"


# Tool 1: Hesap Makinesi
def calculator(expression):
    """Matematik ifadesi hesapla"""
    # Guvenli eval
    allowed = set("0123456789+-*/().%, ")
    if not all(c in allowed for c in expression):
        return "Gecersiz ifade"
    return str(eval(expression))


# Tool 2: Tarih/Saat
def get_datetime(query):
    """Tarih ve saat bilgisi"""
    now = datetime.now()
    if "saat" in query.lower() or "time" in query.lower():
        return f"Saat: {now.strftime('%H:%M:%S')}"
    elif "gun" in query.lower() or "day" in query.lower():
        days_tr = ["Pazartesi","Sali","Carsamba","Persembe","Cuma","Cumartesi","Pazar"]
        return f"Bugun: {days_tr[now.weekday()]}, {now.strftime('%d/%m/%Y')}"
    return f"Tarih: {now.strftime('%d/%m/%Y %H:%M:%S')}"


# Tool 3: Sozluk (Bilgi Bankasi)
knowledge = {
    "python": "Python, Guido van Rossum tarafindan gelistirilmis yuksek seviyeli programlama dilidir. Veri bilimi ve yapay zeka icin en populer dildir.",
    "transformer": "Transformer, 2017'de Google tarafindan gelistirilmis self-attention tabanli derin ogrenme mimarisidir. BERT, GPT gibi modellerin temelidir.",
    "langchain": "LangChain, LLM tabanli uygulamalar gelistirmek icin kullanilan bir Python framework'udur. Chain, Agent ve Memory bilesenleri vardir.",
    "rag": "RAG (Retrieval Augmented Generation), LLM'lerin dis bilgi kaynaklarindan bilgi cekerek daha dogru cevaplar uretmesini saglayan tekniktir.",
    "docker": "Docker, uygulamalari container'lar icinde paketleyen bir platformdur. ML modellerini production'a tasimak icin yaygin kullanilir.",
    "kubernetes": "Kubernetes, container'lari otomatik olarak yoneten bir orkestrasyon platformudur. K8s olarak da bilinir.",
    "mlops": "MLOps, ML modellerinin yasam dongusunu yoneten pratikler butunudur. CI/CD, monitoring ve model registry icerir.",
    "fastapi": "FastAPI, Python ile yuksek performansli API'lar olusturmak icin kullanilan modern bir web framework'udur.",
}

def lookup(query):
    """Bilgi bankasinda arama yap"""
    query_lower = query.lower().strip()
    for key, value in knowledge.items():
        if key in query_lower:
            return value
    return "Bu konu hakkinda bilgi bulunamadi."


# Tool 4: String islemleri
def string_tool(query):
    """Metin islemleri: kelime sayma, buyuk/kucuk harf, ters cevirme"""
    if "kelime say" in query.lower() or "word count" in query.lower():
        text = query.split(":")[-1].strip() if ":" in query else query
        return f"Kelime sayisi: {len(text.split())}"
    elif "buyuk harf" in query.lower() or "upper" in query.lower():
        text = query.split(":")[-1].strip() if ":" in query else query
        return text.upper()
    elif "kucuk harf" in query.lower() or "lower" in query.lower():
        text = query.split(":")[-1].strip() if ":" in query else query
        return text.lower()
    elif "ters" in query.lower() or "reverse" in query.lower():
        text = query.split(":")[-1].strip() if ":" in query else query
        return text[::-1]
    return f"Metin uzunlugu: {len(query)} karakter, {len(query.split())} kelime"


# Tool listesi
tools = [
    Tool("calculator", "Matematik hesaplamalari yapar. Girdi: matematik ifadesi (ornek: '15*7+3')", calculator),
    Tool("datetime", "Tarih ve saat bilgisi verir. Girdi: 'saat', 'gun' veya 'tarih'", get_datetime),
    Tool("lookup", "AI/ML konularinda bilgi arar. Girdi: aranacak konu (ornek: 'python', 'transformer')", lookup),
    Tool("string_tool", "Metin islemleri yapar: kelime sayma, buyuk/kucuk harf. Girdi: islem:metin", string_tool),
]

print("Tanimli Tool'lar:")
for t in tools:
    print(f"  [{t.name}] {t.description}")

In [ ]:
# ============================================
# ADIM 2: ReAct Agent
# ============================================

class ReActAgent:
    """
    ReAct (Reasoning + Acting) Agent
    
    Dongü:
    1. THOUGHT: Durumu degerlendir, ne yapman gerektigini dusun
    2. ACTION: Bir tool sec ve calistir
    3. OBSERVATION: Tool'un sonucunu gor
    4. Tekrarla veya FINAL ANSWER ver
    """
    
    def __init__(self, tools, llm_func, max_steps=5, verbose=True):
        self.tools = {t.name: t for t in tools}
        self.llm = llm_func
        self.max_steps = max_steps
        self.verbose = verbose
        self.history = []
    
    def _build_tool_descriptions(self):
        desc = ""
        for name, tool in self.tools.items():
            desc += f"  - {name}: {tool.description}\n"
        return desc
    
    def _decide_action(self, query, steps_so_far):
        """LLM veya kural tabanli karar verme"""
        query_lower = query.lower()
        
        # Onceki adimlarin sonuclarini kontrol et
        if steps_so_far:
            last_obs = steps_so_far[-1].get("observation", "")
            if last_obs and "Hata" not in last_obs and "bulunamadi" not in last_obs:
                return None, None  # Yeterli bilgi var, final answer ver
        
        # Kural tabanli tool secimi (kucuk LLM icin daha guvenilir)
        # Matematik
        math_patterns = ["+", "-", "*", "/", "hesapla", "kac eder", "toplam", "carpim", "bolum", "karekoku"]
        if any(p in query_lower for p in math_patterns):
            # Matematik ifadesini cikar
            expr = re.findall(r'[\d+\-*/().% ]+', query)
            if expr:
                longest = max(expr, key=len).strip()
                if longest and any(c.isdigit() for c in longest):
                    return "calculator", longest
        
        # Tarih/Saat
        time_patterns = ["saat", "tarih", "gun", "zaman", "time", "date", "day"]
        if any(p in query_lower for p in time_patterns):
            return "datetime", query
        
        # Metin islemleri
        str_patterns = ["kelime say", "buyuk harf", "kucuk harf", "ters cevir", "word count", "upper", "lower"]
        if any(p in query_lower for p in str_patterns):
            return "string_tool", query
        
        # Bilgi arama
        for key in knowledge.keys():
            if key in query_lower:
                return "lookup", query
        
        # Genel bilgi arama
        info_patterns = ["nedir", "ne demek", "acikla", "anlat", "bilgi", "what is"]
        if any(p in query_lower for p in info_patterns):
            return "lookup", query
        
        return None, None
    
    def _generate_answer(self, query, steps):
        """Toplanan bilgilerle final cevap uret"""
        if not steps:
            # Tool kullanilmadi, direkt LLM'e sor
            return self.llm(f"Answer briefly: {query}")
        
        # Son observation'i cevap olarak kullan
        observations = [s["observation"] for s in steps if "observation" in s]
        context = "\n".join(observations)
        
        prompt = f"""Based on this information, answer the question.

Information: {context}

Question: {query}

Answer briefly and clearly:"""
        
        return self.llm(prompt)
    
    def run(self, query):
        """Agent'i calistir"""
        if self.verbose:
            print(f"\n{'='*60}")
            print(f"SORU: {query}")
            print(f"{'='*60}")
        
        steps = []
        
        for step in range(self.max_steps):
            # Karar ver
            tool_name, tool_input = self._decide_action(query, steps)
            
            if tool_name is None:
                # Final answer
                break
            
            # Action
            if self.verbose:
                print(f"\n  Adim {step+1}:")
                print(f"  THOUGHT: '{tool_name}' aracini kullanmaliyim")
                print(f"  ACTION: {tool_name}('{tool_input}')")
            
            # Tool calistir
            if tool_name in self.tools:
                observation = self.tools[tool_name].run(tool_input)
            else:
                observation = f"Bilinmeyen tool: {tool_name}"
            
            if self.verbose:
                print(f"  OBSERVATION: {observation}")
            
            steps.append({
                "step": step + 1,
                "tool": tool_name,
                "input": tool_input,
                "observation": observation,
            })
        
        # Final answer
        answer = self._generate_answer(query, steps)
        
        if self.verbose:
            print(f"\n  FINAL ANSWER: {answer}")
            print(f"  (Toplam {len(steps)} tool cagrisi)")
            print(f"{'='*60}")
        
        result = {
            "query": query,
            "answer": answer,
            "steps": steps,
            "num_steps": len(steps),
        }
        self.history.append(result)
        return result


# Agent olustur
agent = ReActAgent(tools=tools, llm_func=ask_llm, max_steps=5, verbose=True)
print("ReAct Agent hazir!")

In [ ]:
# Agent'i test et
test_queries = [
    "125 * 48 + 320 kac eder?",
    "Bugun gunlerden ne?",
    "Transformer nedir?",
    "Python hakkinda bilgi ver",
    "RAG sistemi ne ise yarar?",
    "Kelime say: Yapay zeka gelecegi sekillendiren onemli bir teknolojidir",
]

for q in test_queries:
    agent.run(q)
    print()

---
## BOLUM 3: Custom Tool Olusturma

Kendi araclarini nasil yazacagini ogrenelim.

In [ ]:
# ============================================
# TOOL 1: Birim Cevirici
# ============================================
def unit_converter(query):
    """Birim cevirme: sicaklik, uzunluk, agirlik"""
    query = query.lower().strip()
    
    # Sicaklik
    if "celsius" in query and "fahrenheit" in query:
        num = float(re.findall(r'[\d.]+', query)[0])
        if "celsius" in query.split("fahrenheit")[0]:
            return f"{num}°C = {num * 9/5 + 32:.1f}°F"
        else:
            return f"{num}°F = {(num - 32) * 5/9:.1f}°C"
    
    # Uzunluk
    if "km" in query and "mil" in query or "mile" in query:
        num = float(re.findall(r'[\d.]+', query)[0])
        if "km" in query.split("mil")[0]:
            return f"{num} km = {num * 0.621371:.2f} mil"
        else:
            return f"{num} mil = {num * 1.60934:.2f} km"
    
    # Agirlik
    if "kg" in query and ("lb" in query or "pound" in query):
        num = float(re.findall(r'[\d.]+', query)[0])
        return f"{num} kg = {num * 2.20462:.2f} lb"
    
    return "Desteklenen ceviriler: celsius/fahrenheit, km/mil, kg/lb"


# ============================================
# TOOL 2: Liste Islemleri
# ============================================
def list_operations(query):
    """Liste islemleri: siralama, ortalama, min, max"""
    numbers = [float(x) for x in re.findall(r'-?[\d.]+', query)]
    
    if not numbers:
        return "Sayi bulunamadi"
    
    import statistics
    
    result = f"Sayilar: {numbers}\n"
    result += f"  Toplam  : {sum(numbers):.2f}\n"
    result += f"  Ortalama: {statistics.mean(numbers):.2f}\n"
    result += f"  Min     : {min(numbers):.2f}\n"
    result += f"  Max     : {max(numbers):.2f}\n"
    result += f"  Sirali  : {sorted(numbers)}"
    
    return result


# ============================================
# TOOL 3: JSON Islemleri
# ============================================
def json_tool(query):
    """JSON olusturma ve parse etme"""
    if "olustur" in query.lower() or "create" in query.lower():
        # Basit key-value cikarma
        pairs = re.findall(r'(\w+)\s*[=:]\s*([\w\s]+?)(?:,|$)', query)
        if pairs:
            data = {k.strip(): v.strip() for k, v in pairs}
            return json.dumps(data, ensure_ascii=False, indent=2)
    
    # JSON parse etme
    try:
        json_match = re.search(r'\{.*\}', query, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group())
            result = "JSON icerigi:\n"
            for k, v in data.items():
                result += f"  {k}: {v}\n"
            return result
    except:
        pass
    
    return "JSON islemi yapilamadi. Ornek: 'JSON olustur: isim=Ali, yas=25, sehir=Istanbul'"


# Yeni tool'lari ekle
new_tools = [
    Tool("unit_converter", "Birim cevirir: celsius/fahrenheit, km/mil, kg/lb. Ornek: '100 celsius fahrenheit'", unit_converter),
    Tool("list_ops", "Sayilar uzerinde islem: toplam, ortalama, min, max, siralama. Ornek: '15, 7, 23, 4, 19'", list_operations),
    Tool("json_tool", "JSON olusturma/parse. Ornek: 'JSON olustur: isim=Ali, yas=25'", json_tool),
]

# Agent'a ekle
all_tools = tools + new_tools
agent2 = ReActAgent(tools=all_tools, llm_func=ask_llm, max_steps=5, verbose=True)

# Test
agent2.run("37 celsius kac fahrenheit?")
agent2.run("15, 7, 23, 4, 19, 31, 8 sayilarinin ortalamasi kac?")
agent2.run("JSON olustur: isim=Zeynep, pozisyon=ML Engineer, sehir=Istanbul")

---
## BOLUM 4: LangChain Chain'leri ve LCEL

In [ ]:
# LangChain ile chain olusturma
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LangChain LLM wrapper
langchain_llm = HuggingFacePipeline(pipeline=llm_pipe)

# ============================================
# CHAIN 1: Basit soru-cevap chain
# ============================================
simple_prompt = PromptTemplate.from_template(
    "Answer the following question briefly and clearly: {question}"
)

# LCEL ile chain olustur (| operatoru)
simple_chain = simple_prompt | langchain_llm | StrOutputParser()

# Test
result = simple_chain.invoke({"question": "What is machine learning?"})
print(f"Basit Chain Sonucu: {result}")

In [ ]:
# ============================================
# CHAIN 2: Cok adimli chain - Ozet + Analiz
# ============================================

# Adim 1: Ozetle
summarize_prompt = PromptTemplate.from_template(
    "Summarize the following text in 2 sentences: {text}"
)

# Adim 2: Anahtar kelime cikar
keyword_prompt = PromptTemplate.from_template(
    "Extract 5 keywords from this text: {text}"
)

# Her iki chain
summarize_chain = summarize_prompt | langchain_llm | StrOutputParser()
keyword_chain = keyword_prompt | langchain_llm | StrOutputParser()

# Test metni
test_text = """Transformer mimarisi 2017 yilinda Google arastirmacilari tarafindan
Attention Is All You Need makalesiyle tanitildi. Self-attention mekanizmasi
sayesinde paralel isleme mumkun oldu ve NLP alaninda devrim yaratildi.
BERT, GPT ve T5 gibi modeller transformer temellidir."""

print("COKLU CHAIN ORNEGI")
print("=" * 60)
summary = summarize_chain.invoke({"text": test_text})
print(f"Ozet: {summary}")

keywords = keyword_chain.invoke({"text": test_text})
print(f"Anahtar kelimeler: {keywords}")

In [ ]:
# ============================================
# CHAIN 3: Sequential chain (Zincirleme)
# ============================================

# Adim 1: Soruyu Ingilizceye cevir
translate_prompt = PromptTemplate.from_template(
    "Translate to English: {question}"
)

# Adim 2: Cevapla
answer_prompt = PromptTemplate.from_template(
    "Answer this question: {english_question}"
)

# Adim 3: Turkceye cevir
tr_prompt = PromptTemplate.from_template(
    "Translate to Turkish: {answer}"
)

# Sequential calistir
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def run_sequential_qa(question):
    """Turkce soru -> Ingilizce cevir -> Cevapla -> Turkceye cevir"""
    print(f"Orijinal: {question}")
    
    # Adim 1
    english_q = (translate_prompt | langchain_llm | StrOutputParser()).invoke({"question": question})
    print(f"Ingilizce: {english_q}")
    
    # Adim 2
    answer = (answer_prompt | langchain_llm | StrOutputParser()).invoke({"english_question": english_q})
    print(f"Cevap (EN): {answer}")
    
    # Adim 3
    tr_answer = (tr_prompt | langchain_llm | StrOutputParser()).invoke({"answer": answer})
    print(f"Cevap (TR): {tr_answer}")
    
    return tr_answer

print("SEQUENTIAL CHAIN")
print("=" * 60)
run_sequential_qa("Yapay zeka nedir?")

---
## BOLUM 5: Memory - Konusma Gecmisi

In [ ]:
# ============================================
# Sifirdan Memory Sistemi
# ============================================

class ConversationMemory:
    """Konusma gecmisini yoneten memory sistemi"""
    
    def __init__(self, max_turns=10):
        self.history = []
        self.max_turns = max_turns
        self.summary = ""
    
    def add(self, role, content):
        self.history.append({"role": role, "content": content})
        if len(self.history) > self.max_turns * 2:
            self._compress()
    
    def _compress(self):
        """Eski mesajlari ozetle"""
        old = self.history[:4]
        summary_text = " | ".join([f"{m['role']}: {m['content'][:50]}" for m in old])
        self.summary = f"Onceki konusma ozeti: {summary_text}"
        self.history = self.history[4:]
    
    def get_context(self):
        parts = []
        if self.summary:
            parts.append(self.summary)
        for msg in self.history:
            prefix = "Kullanici" if msg["role"] == "user" else "Asistan"
            parts.append(f"{prefix}: {msg['content']}")
        return "\n".join(parts)
    
    def clear(self):
        self.history = []
        self.summary = ""


class ChatBot:
    """Memory'li chatbot"""
    
    def __init__(self, llm_func, memory=None):
        self.llm = llm_func
        self.memory = memory or ConversationMemory()
    
    def chat(self, message):
        self.memory.add("user", message)
        
        context = self.memory.get_context()
        
        prompt = f"""You are a helpful AI assistant. Use the conversation history to give relevant answers.

Conversation:
{context}

Respond to the last message briefly:"""
        
        response = self.llm(prompt)
        self.memory.add("assistant", response)
        
        return response
    
    def show_history(self):
        print("\nKonusma Gecmisi:")
        print("-" * 40)
        for msg in self.memory.history:
            prefix = "Sen" if msg["role"] == "user" else "Bot"
            print(f"  {prefix}: {msg['content']}")


# Test
bot = ChatBot(ask_llm)

print("CHATBOT (Memory'li)")
print("=" * 60)

messages = [
    "Merhaba, benim adim Ali",
    "Python ogrenmek istiyorum",
    "Hangi kutuphanleri ogrenmem lazim?",
    "Benim adim neydi?",
]

for msg in messages:
    print(f"\nSen: {msg}")
    response = bot.chat(msg)
    print(f"Bot: {response}")

bot.show_history()

---
## BOLUM 6: RAG Agent - Bilgi Bankasi + Agent

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
import numpy as np

# Embedding modeli ve ChromaDB
emb_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

# Bilgi bankasi
rag_collection = chroma_client.create_collection("rag_agent_kb", metadata={"hnsw:space": "cosine"})

docs = {
    "python_basics": "Python programlama dilinde degiskenler, donguler (for, while), kosul yapilari (if/elif/else), fonksiyonlar ve listeler temel kavramlardir. Python dinamik tipli bir dildir, degisken tipleri otomatik belirlenir.",
    "python_libs": "Python veri bilimi kutuphaneleri: NumPy sayisal hesaplamalar, Pandas veri manipulasyonu, Matplotlib gorsellestirme, Scikit-learn makine ogrenmesi, PyTorch derin ogrenme, HuggingFace Transformers NLP icin kullanilir.",
    "ml_basics": "Makine ogrenmesi uc ana kategoriye ayrilir: Denetimli ogrenme (siniflandirma, regresyon), Denetimsiz ogrenme (kumeleme, boyut indirgeme), Takviyeli ogrenme (odul tabanlı karar verme). Scikit-learn baslangic icin idealdir.",
    "dl_basics": "Derin ogrenme yapay sinir aglari ile calisir. Katmanlar: Dense (tam bagli), Conv (konvolusyon), RNN/LSTM (sirasal), Transformer (dikkat). Aktivasyon fonksiyonlari: ReLU, sigmoid, softmax. Loss fonksiyonlari: cross-entropy, MSE.",
    "llm_usage": "Buyuk dil modelleri (LLM) ile calisma: API uzerinden (OpenAI, Anthropic) veya yerel (HuggingFace, Ollama). Prompt engineering, fine-tuning, RAG temel yaklasimlardir. Temperature, top-p, max_tokens onemli parametrelerdir.",
    "career_advice": "AI kariyerine baslamak icin: 1) Python ogrenin 2) Matematik temeli atin (lineer cebir, istatistik) 3) ML/DL kurslarini tamamlayin 4) Kaggle yarismalarindan katilim 5) GitHub'da portfolio olusturun 6) Blog yazip paylasim yapin.",
    "deployment": "Model deployment adimlari: 1) Modeli kaydet (pickle, ONNX) 2) FastAPI ile API olustur 3) Docker ile containerize et 4) Kubernetes veya bulut servisine deploy et 5) Monitoring ekle (Prometheus+Grafana).",
    "data_eng": "Veri muhendisligi araclari: Apache Spark dagitik islemler, Airflow pipeline yonetimi, Kafka streaming, dbt donusturme, Parquet dosya formati, Delta Lake ACID islemler. ETL ve ELT temel yaklasimlardir.",
}

# ChromaDB'ye ekle
doc_texts = list(docs.values())
doc_ids = list(docs.keys())
doc_embs = emb_model.encode(doc_texts).tolist()

rag_collection.add(ids=doc_ids, documents=doc_texts, embeddings=doc_embs)
print(f"RAG bilgi bankasi: {rag_collection.count()} dokuman")

In [ ]:
# RAG Tool
def rag_search(query):
    """Bilgi bankasinda semantic arama"""
    q_emb = emb_model.encode(query).tolist()
    results = rag_collection.query(
        query_embeddings=[q_emb], n_results=2,
        include=["documents", "distances"]
    )
    
    context = "\n\n".join(results["documents"][0])
    similarities = [f"{1-d:.3f}" for d in results["distances"][0]]
    return f"[Benzerlik: {', '.join(similarities)}]\n{context}"


# RAG + Agent birlesmesi
class RAGAgent:
    """RAG destekli akilli agent"""
    
    def __init__(self, tools, rag_func, llm_func, memory=None):
        self.tools = {t.name: t for t in tools}
        self.rag = rag_func
        self.llm = llm_func
        self.memory = memory or ConversationMemory()
    
    def run(self, query, verbose=True):
        if verbose:
            print(f"\nSoru: {query}")
            print("-" * 50)
        
        self.memory.add("user", query)
        steps = []
        
        # 1) RAG'dan bilgi cek
        if verbose:
            print("  [RAG] Bilgi bankasinda araniyor...")
        rag_result = self.rag(query)
        steps.append({"tool": "rag_search", "result": rag_result})
        if verbose:
            print(f"  [RAG] Sonuc: {rag_result[:100]}...")
        
        # 2) Tool gerekli mi kontrol et
        query_lower = query.lower()
        math_needed = any(p in query_lower for p in ["+", "-", "*", "/", "hesapla", "kac"])
        time_needed = any(p in query_lower for p in ["saat", "tarih", "gun"])
        
        if math_needed and "calculator" in self.tools:
            nums = re.findall(r'[\d+\-*/().]+', query)
            if nums:
                expr = max(nums, key=len)
                calc_result = self.tools["calculator"].run(expr)
                steps.append({"tool": "calculator", "result": calc_result})
                if verbose:
                    print(f"  [CALC] {expr} = {calc_result}")
        
        if time_needed and "datetime" in self.tools:
            time_result = self.tools["datetime"].run(query)
            steps.append({"tool": "datetime", "result": time_result})
            if verbose:
                print(f"  [TIME] {time_result}")
        
        # 3) Cevap uret
        all_info = "\n".join([s["result"] for s in steps])
        conversation = self.memory.get_context()
        
        prompt = f"""Based on this information and conversation history, answer the question.

Information:
{all_info}

Conversation:
{conversation}

Answer the last question briefly and helpfully:"""
        
        answer = self.llm(prompt)
        self.memory.add("assistant", answer)
        
        if verbose:
            print(f"\n  Cevap: {answer}")
            print(f"  ({len(steps)} arac kullanildi)")
            print("=" * 50)
        
        return {"query": query, "answer": answer, "steps": steps}


# RAG Agent olustur
rag_agent = RAGAgent(
    tools=all_tools,
    rag_func=rag_search,
    llm_func=ask_llm,
)

# Test
questions = [
    "Python ile veri bilimi icin hangi kutuphaneleri ogrenmem lazim?",
    "Makine ogrenmesi turleri nelerdir?",
    "Bir modeli production'a nasil tasirim?",
    "AI kariyerine nasil baslayabilirim?",
    "Derin ogrenme katman turleri nelerdir?",
]

for q in questions:
    rag_agent.run(q)
    print()

---
## BOLUM 7: Plan-and-Execute Agent

In [ ]:
class PlanAndExecuteAgent:
    """
    Plan-and-Execute Agent:
    1) Once tum adimlari planla
    2) Her adimi sirayla calistir
    3) Sonuclari birlestir
    """
    
    def __init__(self, tools, llm_func):
        self.tools = {t.name: t for t in tools}
        self.llm = llm_func
    
    def plan(self, query):
        """Gorevi alt adimlara bol"""
        query_lower = query.lower()
        steps = []
        
        # Kural tabanli planlama
        if "karsilastir" in query_lower or "farki" in query_lower or "ve" in query_lower:
            topics = re.split(r'\bve\b|\bile\b|,', query_lower)
            for topic in topics:
                topic = topic.strip().rstrip('?').strip()
                if topic and len(topic) > 3:
                    steps.append({"action": "lookup", "input": topic, "purpose": f"'{topic}' hakkinda bilgi topla"})
            steps.append({"action": "synthesize", "input": "all", "purpose": "Bilgileri birlestir ve karsilastir"})
        
        elif any(p in query_lower for p in ["hesapla", "kac", "+", "*"]):
            steps.append({"action": "calculator", "input": query, "purpose": "Hesaplama yap"})
            steps.append({"action": "synthesize", "input": "all", "purpose": "Sonucu ozetle"})
        
        else:
            steps.append({"action": "rag_search", "input": query, "purpose": "Bilgi bankasinda ara"})
            steps.append({"action": "synthesize", "input": "all", "purpose": "Cevap olustur"})
        
        return steps
    
    def execute(self, query, verbose=True):
        plan = self.plan(query)
        
        if verbose:
            print(f"\nSoru: {query}")
            print(f"\nPLAN ({len(plan)} adim):")
            for i, step in enumerate(plan):
                print(f"  {i+1}. [{step['action']}] {step['purpose']}")
            print()
        
        results = []
        for i, step in enumerate(plan):
            if verbose:
                print(f"ADIM {i+1}: {step['purpose']}")
            
            if step["action"] == "synthesize":
                # Tum sonuclari birlestir
                all_results = "\n".join(results)
                prompt = f"Based on this information, answer: {query}\n\nInformation:\n{all_results}"
                answer = self.llm(prompt)
                if verbose:
                    print(f"  -> {answer}")
                results.append(answer)
            
            elif step["action"] == "rag_search":
                result = rag_search(step["input"])
                if verbose:
                    print(f"  -> {result[:80]}...")
                results.append(result)
            
            elif step["action"] in self.tools:
                result = self.tools[step["action"]].run(step["input"])
                if verbose:
                    print(f"  -> {result}")
                results.append(result)
            
            else:
                result = rag_search(step["input"])
                if verbose:
                    print(f"  -> {result[:80]}...")
                results.append(result)
        
        final = results[-1] if results else "Cevap uretilemedi"
        
        if verbose:
            print(f"\nFINAL: {final}")
            print("=" * 50)
        
        return {"query": query, "plan": plan, "results": results, "answer": final}


pae_agent = PlanAndExecuteAgent(tools=all_tools, llm_func=ask_llm)

# Test
pae_agent.execute("Python ve derin ogrenme hakkinda bilgi ver")
print()
pae_agent.execute("(15 * 8) + (23 * 4) hesapla")

---
## BOLUM 8: Multi-Tool Akilli Asistan

In [ ]:
class SmartAssistant:
    """
    Tam kapsamli akilli asistan:
    - RAG bilgi bankasi
    - Hesap makinesi
    - Tarih/saat
    - Birim cevirici
    - Konusma gecmisi
    - Otomatik tool secimi
    """
    
    def __init__(self, tools, rag_func, llm_func):
        self.tools = {t.name: t for t in tools}
        self.rag = rag_func
        self.llm = llm_func
        self.memory = ConversationMemory(max_turns=10)
        self.stats = {"queries": 0, "tool_calls": 0, "rag_calls": 0}
    
    def _route(self, query):
        """Sorguyu dogru tool'a yonlendir"""
        q = query.lower()
        routes = []
        
        if any(p in q for p in ["+","-","*","/","hesapla","kac eder","toplam","carpim"]):
            routes.append("calculator")
        if any(p in q for p in ["saat","tarih","gun","zaman"]):
            routes.append("datetime")
        if any(p in q for p in ["celsius","fahrenheit","km","mil","kg","lb","cevir"]):
            routes.append("unit_converter")
        if any(p in q for p in ["kelime say","buyuk harf","kucuk harf","ters cevir"]):
            routes.append("string_tool")
        if any(p in q for p in ["sayilar","ortalama","min","max","sirala"]):
            routes.append("list_ops")
        if any(p in q for p in ["json","olustur"]) and "json" in q:
            routes.append("json_tool")
        
        # Her zaman RAG'i da ekle (arka plan bilgisi)
        routes.append("rag")
        
        return routes
    
    def ask(self, query):
        self.stats["queries"] += 1
        self.memory.add("user", query)
        
        routes = self._route(query)
        collected_info = []
        tools_used = []
        
        print(f"\nSen: {query}")
        
        for route in routes:
            if route == "rag":
                info = self.rag(query)
                collected_info.append(f"[Bilgi Bankasi] {info}")
                self.stats["rag_calls"] += 1
            elif route in self.tools:
                result = self.tools[route].run(query)
                collected_info.append(f"[{route}] {result}")
                tools_used.append(route)
                self.stats["tool_calls"] += 1
        
        # Cevap uret
        all_info = "\n".join(collected_info)
        conversation = self.memory.get_context()
        
        prompt = f"""You are a helpful assistant. Use the information and conversation to answer.

Information:
{all_info}

Recent conversation:
{conversation}

Give a helpful, brief answer:"""
        
        answer = self.llm(prompt)
        self.memory.add("assistant", answer)
        
        tool_str = f" [{', '.join(tools_used)}]" if tools_used else ""
        print(f"Bot{tool_str}: {answer}")
        
        return answer
    
    def show_stats(self):
        print(f"\nIstatistikler:")
        print(f"  Toplam sorgu  : {self.stats['queries']}")
        print(f"  Tool cagrisi  : {self.stats['tool_calls']}")
        print(f"  RAG cagrisi   : {self.stats['rag_calls']}")


# Asistan olustur
assistant = SmartAssistant(
    tools=all_tools,
    rag_func=rag_search,
    llm_func=ask_llm,
)

# Interaktif sohbet simulasyonu
print("AKILLI ASISTAN SOHBET")
print("=" * 60)

conversation = [
    "Python ogrenmek istiyorum, nereden baslamaliyim?",
    "Makine ogrenmesi icin hangi kutuphane lazim?",
    "42 celsius kac fahrenheit?",
    "Bugun gunlerden ne?",
    "15, 28, 7, 42, 3, 19 sayilarinin ortalamasi?",
    "Bir modeli nasil deploy ederim?",
    "125 * 48 kac eder?",
    "LangChain nedir?",
]

for msg in conversation:
    assistant.ask(msg)

assistant.show_stats()

---
## BOLUM 9: Agent Debugging ve Guvenlik

In [ ]:
class SafeAgent(ReActAgent):
    """Guvenlik ozellikleri ekli agent"""
    
    def __init__(self, tools, llm_func, max_steps=5, max_time=30, blocked_words=None):
        super().__init__(tools, llm_func, max_steps, verbose=True)
        self.max_time = max_time
        self.blocked_words = blocked_words or []
        self.log = []
    
    def run(self, query):
        import time
        start = time.time()
        
        # Guvenlik kontrolu
        for word in self.blocked_words:
            if word.lower() in query.lower():
                msg = f"GUVENLIK: '{word}' iceren sorgu reddedildi."
                print(msg)
                self.log.append({"type": "blocked", "query": query, "reason": msg})
                return {"query": query, "answer": "Bu sorgu guvenlik nedeniyle reddedildi.", "blocked": True}
        
        # Zaman kontrolu
        self.log.append({"type": "start", "query": query, "time": time.time()})
        
        result = super().run(query)
        
        elapsed = time.time() - start
        if elapsed > self.max_time:
            print(f"UYARI: Sorgu {elapsed:.1f}s surdu (limit: {self.max_time}s)")
        
        self.log.append({
            "type": "complete",
            "query": query,
            "steps": result.get("num_steps", 0),
            "time": round(elapsed, 2)
        })
        
        return result
    
    def show_log(self):
        print("\nAgent Log:")
        print("-" * 50)
        for entry in self.log:
            if entry["type"] == "blocked":
                print(f"  [BLOCKED] {entry['query'][:40]}... -> {entry['reason']}")
            elif entry["type"] == "complete":
                print(f"  [OK] {entry['query'][:40]}... ({entry['steps']} adim, {entry['time']}s)")


# Safe agent olustur
safe_agent = SafeAgent(
    tools=all_tools,
    llm_func=ask_llm,
    max_steps=5,
    max_time=30,
    blocked_words=["rm -rf", "delete all", "drop table"]
)

# Normal sorgular
safe_agent.run("Python nedir?")
safe_agent.run("15 * 8 hesapla")

# Engellenen sorgu
safe_agent.run("rm -rf / komutu ne yapar?")

safe_agent.show_log()

---
## Egzersizler

### Egzersiz 1 (Kolay)
Yeni bir tool yaz: "password_generator" - belirtilen uzunlukta rastgele sifre uretsin.
Agent'a ekle ve test et.

In [ ]:
# Senin cozumun


### Egzersiz 2 (Orta)
RAG Agent'ina 5 yeni dokuman ekle (ornegin Docker, Git, API tasarimi, veritabani, guvenlik).
Yeni sorularla test et ve sonuclari degerlendir.

In [ ]:
# Senin cozumun


### Egzersiz 3 (Zor)
Bir "Kod Asistani" agent'i yaz:
1. Kullanicinin sorusunu analiz et
2. Gerekirse bilgi bankasinda ara
3. Basit kod ornegi uret (string template ile)
4. Kodu acikla
5. Konusma gecmisini tut

In [ ]:
# Senin cozumun


---
## Egzersiz Cozumleri

In [ ]:
# COZUM 1 - Password Generator
import random
import string

def password_generator(query):
    """Rastgele sifre uret"""
    nums = re.findall(r'\d+', query)
    length = int(nums[0]) if nums else 12
    length = max(6, min(length, 50))
    
    chars = string.ascii_letters + string.digits + "!@#$%&*"
    password = ''.join(random.choice(chars) for _ in range(length))
    
    # En az 1 buyuk, 1 kucuk, 1 rakam, 1 ozel karakter
    password = list(password)
    password[0] = random.choice(string.ascii_uppercase)
    password[1] = random.choice(string.ascii_lowercase)
    password[2] = random.choice(string.digits)
    password[3] = random.choice("!@#$%&*")
    random.shuffle(password)
    
    return f"Sifre ({length} karakter): {''.join(password)}"

pw_tool = Tool("password", "Rastgele sifre uretir. Girdi: uzunluk (ornek: '16 karakter')", password_generator)

# Agent'a ekle ve test et
test_agent = ReActAgent(tools=all_tools + [pw_tool], llm_func=ask_llm)
# Manuel test
print(password_generator("16 karakter"))
print(password_generator("8"))
print(password_generator("24 karakterlik guclu sifre"))

In [ ]:
# COZUM 2 - Yeni dokumanlar
new_docs = {
    "docker_guide": "Docker container teknolojisidir. Dockerfile ile image olusturulur. docker build, docker run, docker-compose temel komutlardir. Multi-stage build ile kucuk image'lar olusturulur. Volume'lar veri kaliciligi saglar.",
    "git_basics": "Git versiyon kontrol sistemidir. Temel komutlar: git init, add, commit, push, pull, branch, merge. GitHub ve GitLab uzak depo servisleridir. .gitignore ile dosyalar haric tutulur. Pull request code review icin kullanilir.",
    "api_design": "REST API tasarimi: HTTP metodlari GET, POST, PUT, DELETE. Status kodlari 200 basari, 404 bulunamadi, 500 sunucu hatasi. JSON veri formati standart. Authentication icin JWT token veya API key kullanilir. Rate limiting onemlidir.",
    "database_101": "Veritabani turleri: SQL (PostgreSQL, MySQL) iliskisel veri icin, NoSQL (MongoDB, Redis) esnek yapili veri icin. ORM araclari: SQLAlchemy Python icin. ACID ozellikleri veri butunlugunu saglar. Index'ler sorgu performansini arttirir.",
    "security_basics": "AI guvenlik: Prompt injection saldirilarina karsi girdi dogrulama. API anahtarlarini environment variable'da sakla. Rate limiting ile kotu kullanimi onle. Model ciktisini filtrele. OWASP LLM Top 10 guvenlik riskleri rehberdir.",
}

new_embs = emb_model.encode(list(new_docs.values())).tolist()
rag_collection.add(
    ids=list(new_docs.keys()),
    documents=list(new_docs.values()),
    embeddings=new_embs,
)
print(f"Bilgi bankasi guncellendi: {rag_collection.count()} dokuman")

for q in ["Docker nedir?", "Git komutlari nelerdir?", "API guvenlik"]:
    print(f"\nSoru: {q}")
    result = rag_search(q)
    print(f"Cevap: {result[:120]}...")

In [ ]:
# COZUM 3 - Kod Asistani
code_templates = {
    "for_loop": "# For dongusu ornegi\nfor i in range(10):\n    print(f'Eleman: {i}')",
    "function": "# Fonksiyon ornegi\ndef hesapla(a, b):\n    return a + b\n\nsonuc = hesapla(5, 3)\nprint(sonuc)",
    "list_comp": "# List comprehension\nsayilar = [1, 2, 3, 4, 5]\nkareler = [x**2 for x in sayilar]\nprint(kareler)",
    "class": "# Sinif ornegi\nclass Hayvan:\n    def __init__(self, isim):\n        self.isim = isim\n    def ses_cikar(self):\n        print(f'{self.isim} ses cikariyor')\n\nkedi = Hayvan('Tekir')\nkedi.ses_cikar()",
    "file_io": "# Dosya okuma/yazma\nwith open('veri.txt', 'w') as f:\n    f.write('Merhaba')\n\nwith open('veri.txt', 'r') as f:\n    icerik = f.read()\nprint(icerik)",
    "dict": "# Sozluk kullanimi\nogrenci = {'ad': 'Ali', 'yas': 25, 'bolum': 'Bilgisayar'}\nfor anahtar, deger in ogrenci.items():\n    print(f'{anahtar}: {deger}')",
    "try_except": "# Hata yonetimi\ntry:\n    sonuc = 10 / 0\nexcept ZeroDivisionError:\n    print('Sifira bolme hatasi!')\nexcept Exception as e:\n    print(f'Hata: {e}')\nfinally:\n    print('Islem bitti')",
}

class CodeAssistant:
    def __init__(self, rag_func, llm_func):
        self.rag = rag_func
        self.llm = llm_func
        self.memory = ConversationMemory()
        self.templates = code_templates
    
    def find_template(self, query):
        q = query.lower()
        for key in self.templates:
            if key.replace("_", " ") in q or key in q:
                return key, self.templates[key]
        
        keywords = {"dongu": "for_loop", "loop": "for_loop", "fonksiyon": "function",
                    "function": "function", "def": "function", "sinif": "class",
                    "class": "class", "dosya": "file_io", "file": "file_io",
                    "sozluk": "dict", "dict": "dict", "hata": "try_except",
                    "try": "try_except", "list": "list_comp"}
        
        for kw, template_key in keywords.items():
            if kw in q:
                return template_key, self.templates[template_key]
        return None, None
    
    def ask(self, query):
        self.memory.add("user", query)
        print(f"\nSen: {query}")
        
        # 1) RAG'dan bilgi al
        info = self.rag(query)
        
        # 2) Kod sablonu bul
        template_name, code = self.find_template(query)
        
        # 3) Cevap olustur
        response = ""
        if code:
            explanation = self.llm(f"Explain briefly what this code does: {code[:100]}")
            response = f"Iste bir '{template_name}' ornegi:\n\n{code}\n\nAciklama: {explanation}"
        else:
            response = self.llm(f"Based on this context, answer: {query}\nContext: {info[:200]}")
        
        self.memory.add("assistant", response)
        print(f"\nKod Asistani:\n{response}")
        print("-" * 50)
        return response


code_bot = CodeAssistant(rag_func=rag_search, llm_func=ask_llm)

code_bot.ask("Python'da for dongusu nasil yazilir?")
code_bot.ask("Bir sinif nasil olusturulur?")
code_bot.ask("Hata yonetimi nasil yapilir?")
code_bot.ask("List comprehension ornegi goster")

---
## Gunun Ozeti

### Bugün Ogrendiklerimiz:

1. **ReAct Agent** sifirdan: Thought -> Action -> Observation dongusu
2. **Custom Tool**: Kendi araclarini yazma (calculator, datetime, converter)
3. **LangChain Chain**: LCEL ile zincirleme islemler
4. **Memory**: Konusma gecmisi yonetimi (buffer, window, summary)
5. **RAG Agent**: Bilgi bankasi + agent birlesmesi
6. **Plan-and-Execute**: Once planla, sonra calistir
7. **Multi-tool Asistan**: Birden fazla araci akilli kullanan sistem
8. **Agent Safety**: Guvenlik, loglama, zaman limiti

### Yarin (Gun 7): LangChain & LlamaIndex ile Otonom AI Ajanlari Gelistirme
(Devam - ileri agent teknikleri)